# Finned-pipe models and their submodels

PyTES models are fixed compositions of submodels: a fluid, a storage and a coupling, plus the
device-level energy. This notebook shows how to inspect them, run a discharge, and check the
two invariants every model meets: exact energy conservation and the state of charge.

In [1]:
import numpy as np
import pybamm

import pytes

model = pytes.SharpFrontModel(operating_mode=None)
for name, submodel in model.submodels.items():
    print(f"{name}: {type(submodel).__name__}")
    print("    requires:", submodel.requires)
    print("    provides:", submodel.provides)

fluid: AdvectedFluid
    requires: ('Heat transfer fluid heat gain per unit length [W.m-1]', 'Initial uniform temperature [K]')
    provides: ('Heat transfer fluid temperature [K]', 'Inlet temperature [K]', 'Outlet temperature [K]', 'Inlet specific enthalpy [J.kg-1]', 'Outlet specific enthalpy [J.kg-1]', 'Mass flow rate [kg.s-1]', 'Heat transfer fluid thermal energy [J]')
storage: EnthalpyStorage
    requires: ('Initial uniform temperature [K]', 'Initial uniform liquid fraction')
    provides: ('Phase-change material enthalpy [J.m-3]', 'Phase-change material enthalpy rate of change [W.m-3]', 'Phase-change material temperature [K]', 'Phase-change material surface temperature [K]', 'Phase-change material surface enthalpy [J.m-3]', 'Liquid fraction', 'Phase', 'Average liquid fraction', 'Phase-change material thermal energy [J]')
coupling: LedgerWallCoupling
    requires: ('Heat transfer fluid temperature [K]', 'Phase-change material enthalpy rate of change [W.m-3]', 'Phase-change material

The model is an ordinary PyBaMM model, so `pybamm.Simulation` applies directly. By default
the battery starts fully charged (uniform at the upper temperature limit, 70 °C) and is
discharged with water at 10 °C.

In [2]:
sim = pybamm.Simulation(model)
solution = sim.solve([0, 3600], t_interp=np.linspace(0, 3600, 121))
sim.plot()

interactive(children=(FloatSlider(value=0.0, description='t', max=1.0, step=0.01), Output()), _dom_classes=('w…

Power is `mdot (h_out - h_in)`, positive on discharge. The change in total thermal energy
(storage plus fluid) is minus the discharged energy, to round-off.

In [3]:
E = solution["Total thermal energy [J]"].entries
discharged = solution["Discharged energy [J]"].entries
print("max conservation error [J]:", np.max(np.abs(E - E[0] + discharged)))
print("discharged energy [kWh]:", solution["Discharged energy [kWh]"].entries[-1])
print("thermal capacity [kWh]:", solution["Thermal capacity [J]"].entries[0] / 3.6e6)

max conservation error [J]: 1.30385160446167e-07
discharged energy [kWh]: 7.027502416260745
thermal capacity [kWh]: 7.027538449357847


The state of charge is energy-based: 1 and 0 are the uniform states at the upper and lower
temperature limits. The initial state is set by `"Initial state of charge"`.

In [4]:
parameter_values = model.default_parameter_values
parameter_values["Initial state of charge"] = 0.5
half = pybamm.Simulation(model, parameter_values=parameter_values).solve([0, 600])
print("initial SOC:", half["State of charge"].entries[0])
print(
    "initial uniform temperature [degC]:",
    half["Initial uniform temperature [degC]"].entries[0],
)
print(
    "initial uniform liquid fraction:",
    half["Initial uniform liquid fraction"].entries[0],
)

initial SOC: 0.49999999999999983
initial uniform temperature [degC]: 58.0
initial uniform liquid fraction: 0.24870258665527956


The mush model resolves the fins and the PCM between them, and has the same outputs, so
the two compare one to one.

In [5]:
mush = pytes.MushModel(operating_mode=None)
mush_solution = pybamm.Simulation(
    mush, var_pts={mush.x: 10, mush.y: 20, mush.z: 20}
).solve([0, 3600], t_interp=np.linspace(0, 3600, 121))
pybamm.dynamic_plot(
    [solution, mush_solution],
    ["Outlet temperature [degC]", "Power [kW]", "State of charge"],
)

interactive(children=(FloatSlider(value=0.0, description='t', max=1.0, step=0.01), Output()), _dom_classes=('w…